# 01 — Training: MDP Baseline vs MLP-Bellman vs MLP-Full
**Paper:** *Markov Logic Process: Augmenting Reinforcement Learning with Symbolic
Association-Rule Reasoning via the Logos Module*
**Authors:** Saiyam Jain · Swaroop Bhowmik · Dipanjan Choudhury · Santosh Kumar Sahoo

### What this notebook does
Trains three agent variants across two LunarLander environments × 5 seeds (600 episodes each).
Saves Q-network weights and per-seed results for downstream evaluation and video generation.

### Session-safety features
- ✅ Per-seed checkpoints — resumes from any interruption automatically
- ✅ Wall-clock budget guard — stops ~30 min before Kaggle's 12-hr limit
- ✅ Best-model export — picks the highest `eval_mean` seed per (agent, env)

## 0 · Install & Imports

In [ ]:
# ── 0a. System dependencies ───────────────────────────────────────────────────
import subprocess, sys

def apt(pkg):
    subprocess.check_call(
        ['apt-get', 'install', '-y', '-q', pkg],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )

def pip(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

apt('swig')
apt('build-essential')
pip('gymnasium[box2d]>=0.29.0')
pip('mlxtend>=0.23.0')
pip('stable-baselines3')
pip('scipy')

print("✅ dependencies installed")

In [ ]:
# ── 0b. Standard imports ──────────────────────────────────────────────────────
import os, sys, json, time, random, warnings
from pathlib import Path
from collections import deque
from concurrent.futures import ThreadPoolExecutor
import threading

import numpy as np
import pandas as pd
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
from scipy import stats as scipy_stats
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import matplotlib; import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

# Try importing from src/ (local dev); fall back to inline definitions below
sys.path.insert(0, '/kaggle/working')  # notebook root
try:
    sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ''))
    from src.utils.checkpoint import make_dirs, save_seed_result, load_all_seed_results, \
                                      is_done, save_model, save_best_model, print_progress
    from src.utils.stats import compute_ma, episodes_to_solve, summary_stats
    USING_SRC = True
    print("✅ imported from src/")
except ImportError:
    USING_SRC = False
    print("⚠️  src/ not found — using inline definitions below")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}  |  Gym: {gym.__version__}")
try:
    _t = gym.make('LunarLander-v3'); _t.close()
    print("✅ LunarLander-v3 OK")
except Exception as e:
    print(f"❌ LunarLander error: {e}")

## 1 · Configuration

In [ ]:
# ── Experiment hyperparameters (matches paper exactly) ────────────────────────
SEEDS     = [42, 123, 456, 789, 1337]
N_EP      = 600    # episodes per seed
EVAL_EP   = 50     # greedy evaluation episodes after training
SOLVE_THR = 200    # MA-100 threshold

AGENT_TYPES = ['MDP', 'MLP-Bellman', 'MLP-Full']

EXPERIMENT_ENVS = {
    'll_std': {
        'display_name': 'LunarLander (standard)',
        'kwargs': {},
    },
    'll_wind': {
        'display_name': 'LunarLander (wind + turbulence)',
        'kwargs': {'enable_wind': True, 'wind_power': 15.0, 'turbulence_power': 1.5},
    },
}

# DQN backbone
DQN_CFG = dict(
    hidden=(64, 64),
    lr=1e-3,
    gamma=0.99,
    batch_size=64,
    replay_capacity=10_000,
    target_update_freq=100,
    eps_start=1.0,
    eps_end=0.05,
    eps_decay_episodes=300,
)

# Logos module
LOGOS_CFG = dict(
    lambda_scale=1.5,
    window_size=2_000,
    mine_interval=2_000,
    min_support=0.20,
    min_confidence=0.80,
    persistence_cap=5,
    deduction_threshold=0.90,
)

# Kaggle session budget (seconds before 12-hr Kaggle limit)
WALL_BUDGET = 11.5 * 3600
SESSION_START = time.time()

def over_budget():
    return (time.time() - SESSION_START) > WALL_BUDGET

print("Configuration loaded ✅")

## 2 · Checkpoint System

In [ ]:
# Inline checkpoint utilities (used if src/ import failed above)
if not USING_SRC:
    BASE_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
    PATHS = {
        'ckpt':   BASE_DIR / 'checkpoints',
        'seeds':  BASE_DIR / 'checkpoints' / 'seed_ckpts',
        'models': BASE_DIR / 'checkpoints' / 'models',
        'figs':   BASE_DIR / 'checkpoints' / 'figures',
    }
    for p in PATHS.values():
        p.mkdir(parents=True, exist_ok=True)

    def _js(o):
        if isinstance(o,(frozenset,set)): return sorted(list(o))
        if isinstance(o,np.ndarray): return o.tolist()
        if isinstance(o,(np.integer,)): return int(o)
        if isinstance(o,(np.floating,)): return float(o)
        raise TypeError(type(o).__name__)

    def save_seed_result(paths, exp_key, agent, seed, result):
        p = paths['seeds'] / f"{exp_key}__{agent.replace('-','_')}__seed{seed}.json"
        with open(p,'w') as f: json.dump(result,f,default=_js)
        log = {'ts':time.strftime('%Y-%m-%dT%H:%M:%S'),'exp_key':exp_key,'agent':agent,'seed':seed,**{k:result.get(k) for k in ['final_ma100','eval_mean','ep_to_solve']}}
        with open(paths['ckpt']/'run_log.jsonl','a') as f: f.write(json.dumps(log,default=_js)+'\n')

    def is_done(paths, exp_key, agent, seed):
        return (paths['seeds']/f"{exp_key}__{agent.replace('-','_')}__seed{seed}.json").exists()

    def load_all_seed_results(paths):
        out = {}
        for p in sorted(paths['seeds'].glob('*.json')):
            parts = p.stem.split('__')
            if len(parts)!=3: continue
            ek, sa, sp = parts
            at = sa.replace('_','-'); seed = int(sp.replace('seed',''))
            with open(p) as f: res = json.load(f)
            out.setdefault(ek,{}).setdefault(at,{})[seed] = res
        return out

    def save_model(paths, sd, agent, env_key, seed, tag=''):
        suf = f'_{tag}' if tag else ''
        fn = f"{agent.replace('-','_')}_{env_key}_seed{seed}{suf}.pt"
        torch.save(sd, paths['models']/fn)

    def save_best_model(paths, sd, agent, env_key):
        torch.save(sd, paths['models']/f"{agent.replace('-','_')}_{env_key}_best.pt")

    def compute_ma(rewards, w=100):
        arr=np.array(rewards,float)
        if len(arr)<w: return np.full(len(arr),np.nan)
        return np.convolve(arr,np.ones(w)/w,mode='valid')

    def episodes_to_solve(rewards,thr=200,w=100):
        ma=compute_ma(rewards,w)
        idx=np.where(ma>=thr)[0]
        return float(idx[0]+w) if len(idx) else float('inf')

    def summary_stats(vals):
        a=np.array([v for v in vals if np.isfinite(v)],float)
        if not len(a): return dict(mean=None,std=None,min=None,max=None)
        return dict(mean=float(a.mean()),std=float(a.std(ddof=1)),min=float(a.min()),max=float(a.max()),n=len(a))

    def print_progress(ar, eks, ats, seeds):
        total=len(eks)*len(ats)*len(seeds)
        done=sum(1 for ek in eks for at in ats for s in seeds if s in ar.get(ek,{}).get(at,{}))
        print(f"Progress: {done}/{total}")
        for ek in eks:
            print(f"  {ek}:")
            for at in ats:
                done_s=sorted(ar.get(ek,{}).get(at,{}).keys())
                rem=[s for s in seeds if s not in done_s]
                print(f"    {at:<15} {'✅' if not rem else '⏳ '+str(rem)}")

    PATHS = PATHS
else:
    PATHS = make_dirs()

ALL_RESULTS = load_all_seed_results(PATHS)
print_progress(ALL_RESULTS, list(EXPERIMENT_ENVS.keys()), AGENT_TYPES, SEEDS)

## 3 · Network, Replay Buffer & Feature Discretiser

In [ ]:
# ── Q-Network ─────────────────────────────────────────────────────────────────
class QNetwork(nn.Module):
    def __init__(self, obs_dim, n_actions, hidden=(64,64)):
        super().__init__()
        layers, in_d = [], obs_dim
        for h in hidden:
            layers += [nn.Linear(in_d,h), nn.ReLU()]; in_d=h
        layers.append(nn.Linear(in_d,n_actions))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

# ── Replay Buffer ─────────────────────────────────────────────────────────────
class ReplayBuffer:
    def __init__(self, cap=10_000): self._b=deque(maxlen=cap)
    def push(self,o,a,r,no,d): self._b.append((o,a,r,no,d))
    def sample(self,bs):
        batch=random.sample(self._b,bs)
        o,a,r,no,d=zip(*batch)
        return (np.array(o,np.float32),np.array(a,np.int64),
                np.array(r,np.float32),np.array(no,np.float32),np.array(d,np.float32))
    def __len__(self): return len(self._b)

# ── Feature Discretiser ───────────────────────────────────────────────────────
FEAT_NAMES  = ['x','y','x_vel','y_vel','angle','ang_vel']
BIN_LABELS  = ['Low','Med','High']
CONT_IDX    = [0,1,2,3,4,5]

class FeatureDiscretiser:
    """Hybrid EW+EF discretiser (Section IV-B)."""
    def __init__(self):
        self._buf=[]; self._fitted=False
        # LunarLander heuristic fallback ranges
        self._min = np.array([-1.5,-0.5,-2.5,-2.5,-3.14,-5.0])
        self._max = np.array([ 1.5, 1.5, 2.5, 2.5, 3.14, 5.0])
        self._q33 = self._min+(self._max-self._min)/3
        self._q66 = self._min+2*(self._max-self._min)/3

    def partial_fit(self, obs):
        self._buf.append(np.asarray(obs)[CONT_IDX])
        if len(self._buf)>=200:
            arr=np.stack(self._buf)
            self._min=np.minimum(self._min,arr.min(0))
            self._max=np.maximum(self._max,arr.max(0))
            self._q33=np.percentile(arr,33,axis=0)
            self._q66=np.percentile(arr,66,axis=0)
            self._fitted=True; self._buf.clear()

    def transform(self, obs):
        c=np.asarray(obs)[CONT_IDX]
        ew=np.clip(((c-self._min)/(self._max-self._min+1e-8)*3).astype(int),0,2)
        ef=np.zeros(6,int)
        ef[c>=self._q33]=1; ef[c>=self._q66]=2
        b=np.clip(np.round((ew+ef)/2).astype(int),0,2)
        return [(FEAT_NAMES[i],BIN_LABELS[b[i]]) for i in range(6)]

    def bin_reward(self,r): return 'Low' if r<-50 else ('High' if r>50 else 'Med')

    def itemset(self,obs,action,reward=None):
        items=[f'{n}:{l}' for n,l in self.transform(obs)]
        items.append(f'action:{action}')
        if reward is not None: items.append(f'reward:{self.bin_reward(reward)}')
        return items

print("Network, ReplayBuffer, FeatureDiscretiser defined ✅")

## 4 · Logos Module (Asynchronous Apriori Miner)

In [ ]:
class LogosModule:
    """
    Full Logos module L = (D, B, F, τ) — Section IV.
    Mines Apriori rules asynchronously and exposes:
        phi(obs, action)           → scalar Φ_L(s,a)
        phi_all(obs)               → array of Φ_L for all actions
        explain(obs, action)       → (phi, [matched rule dicts])
    """
    def __init__(self, cfg=None, n_actions=4):
        c = cfg or LOGOS_CFG
        self.lam   = c['lambda_scale']
        self.win   = c['window_size']
        self.delta = c['mine_interval']
        self.sigma = c['min_support']
        self.kappa = c['min_confidence']
        self.pcap  = c['persistence_cap']
        self.dth   = c['deduction_threshold']
        self.n_act = n_actions

        self._disc = FeatureDiscretiser()
        self._window = deque(maxlen=self.win)
        self._step = 0

        self._rules = []
        self._persist = {}
        self._UL = 0.0
        self._lock = threading.Lock()
        self._executor = ThreadPoolExecutor(max_workers=1)
        self._busy = False

    # ── Ingestion ────────────────────────────────────────────────────────────
    def add_experience(self, obs, action, reward):
        self._disc.partial_fit(obs)
        self._window.append(self._disc.itemset(obs,action,reward))
        self._step += 1
        if self._step % self.delta == 0 and len(self._window)>=50 and not self._busy:
            self._busy=True
            self._executor.submit(self._mine_bg)

    def _mine_bg(self):
        try:
            snap=list(self._window)
            new_rules=self._run_apriori(snap)
            self._update(new_rules)
        finally:
            self._busy=False

    def _run_apriori(self, transactions):
        if len(transactions)<20: return []
        te=TransactionEncoder()
        try: arr=te.fit_transform(transactions)
        except: return []
        df=pd.DataFrame(arr,columns=te.columns_)
        try: freq=apriori(df,min_support=self.sigma,use_colnames=True,verbose=0)
        except: return []
        if freq.empty: return []
        try: rules_df=association_rules(freq,metric='confidence',min_threshold=self.kappa)
        except: return []
        if rules_df.empty: return []
        valid=[]
        for _,row in rules_df.iterrows():
            ant,con=row['antecedents'],row['consequents']
            ant_nr={i for i in ant if not i.startswith('reward:')}
            if not ant_nr: continue  # Fix 2
            if not any(i.startswith('action:') or i.startswith('reward:') for i in con): continue  # Fix 3
            valid.append({'ant':ant,'con':con,'conf':float(row['confidence']),'supp':float(row['support']),'pers':0})
        return valid

    def _rule_key(self,r): return ','.join(sorted(r['ant']))+'->'+','.join(sorted(r['con']))

    def _update(self,new_rules):
        new_k={self._rule_key(r):r for r in new_rules}
        upd={}
        for k,r in new_k.items():
            upd[k]=self._persist.get(k,0)+1
            r['pers']=upd[k]
        ul=sum(1 for v in upd.values() if v>=3)/len(new_k) if new_k else 0.0
        with self._lock:
            self._rules=list(new_k.values()); self._persist=upd; self._UL=ul

    # ── Potential ────────────────────────────────────────────────────────────
    def _compute(self, obs, action, return_rules=False):
        with self._lock:
            rules=list(self._rules); UL=self._UL
        if not rules: return 0.0,[]
        items=set(self._disc.itemset(obs,action))
        phi=0.0; matched=[]
        for r in rules:
            ant_s=frozenset(i for i in r['ant'] if not i.startswith('reward:'))
            if ant_s.issubset(items):
                contrib=r['conf']*min(r['pers'],self.pcap)
                phi+=contrib
                if return_rules:
                    matched.append({
                        'antecedents_str':' ∧ '.join(sorted(ant_s)),
                        'consequents_str':' ∨ '.join(sorted(r['con'])),
                        'confidence':r['conf'],
                        'persistence':r['pers'],
                        'contribution':self.lam*UL*contrib,
                    })
        phi*=UL
        if return_rules:
            matched.sort(key=lambda x:x['contribution'],reverse=True)
        return phi, matched[:5]

    def phi(self,obs,action):
        v,_=self._compute(obs,action); return self.lam*v

    def phi_all(self,obs):
        return np.array([self.phi(obs,a) for a in range(self.n_act)])

    def explain(self,obs,action):
        v,rules=self._compute(obs,action,return_rules=True); return self.lam*v,rules

    def deduction_value(self,obs,q_vals):
        if self._UL>self.dth: return self.phi_all(obs)
        return q_vals

    @property
    def UL(self): return self._UL
    @property
    def n_rules(self): return len(self._rules)
    def shutdown(self): self._executor.shutdown(wait=False)

print("LogosModule defined ✅")

## 5 · Agent Classes

In [ ]:
class DQNAgent:
    """MDP baseline — standard DQN, no Logos (λ=0)."""
    def __init__(self, obs_dim=8, n_actions=4, cfg=None, device=None):
        c=cfg or DQN_CFG
        self.n_act=n_actions; self.gamma=c['gamma']
        self.bs=c['batch_size']; self.tuf=c['target_update_freq']
        self.eps_start=c['eps_start']; self.eps_end=c['eps_end']
        self.eps_dec=c['eps_decay_episodes']
        self.dev=device or DEVICE
        self.q=QNetwork(obs_dim,n_actions,c['hidden']).to(self.dev)
        self.qt=QNetwork(obs_dim,n_actions,c['hidden']).to(self.dev)
        self.qt.load_state_dict(self.q.state_dict()); self.qt.eval()
        self.opt=optim.Adam(self.q.parameters(),lr=c['lr'])
        self.buf=ReplayBuffer(c['replay_capacity'])
        self._step=0; self._ep=0; self.losses=[]

    @property
    def epsilon(self):
        frac=min(self._ep/max(self.eps_dec,1),1.0)
        return self.eps_start+frac*(self.eps_end-self.eps_start)

    def observe(self,o,a,r,no,d): self.buf.push(o,a,r,no,d)

    def select_action(self,obs,exploit=False):
        if not exploit and random.random()<self.epsilon: return random.randrange(self.n_act)
        t=torch.tensor(obs,dtype=torch.float32,device=self.dev).unsqueeze(0)
        with torch.no_grad(): return int(self.q(t).squeeze(0).argmax().item())

    def _td_target(self,r_t,no_t,d_t,no_np=None,a_np=None):
        with torch.no_grad(): qn=self.qt(no_t).max(1)[0]
        return r_t+self.gamma*qn*(1-d_t)

    def update(self):
        if len(self.buf)<self.bs: return None
        o,a,r,no,d=self.buf.sample(self.bs)
        o_t=torch.tensor(o,device=self.dev); a_t=torch.tensor(a,device=self.dev)
        r_t=torch.tensor(r,device=self.dev); no_t=torch.tensor(no,device=self.dev)
        d_t=torch.tensor(d,device=self.dev)
        qp=self.q(o_t).gather(1,a_t.unsqueeze(1)).squeeze(1)
        qt=self._td_target(r_t,no_t,d_t,no,a)
        loss=nn.functional.mse_loss(qp,qt)
        self.opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(self.q.parameters(),10.0); self.opt.step()
        self._step+=1
        if self._step%self.tuf==0: self.qt.load_state_dict(self.q.state_dict())
        self.losses.append(loss.item()); return loss.item()

    def state_dict_export(self):
        return {'q_net':self.q.state_dict(),'target_net':self.qt.state_dict(),
                'episode':self._ep,'step':self._step}


class MLPBellmanAgent(DQNAgent):
    """MLP-Bellman: Logos shaping in TD target only."""
    def __init__(self,*args,logos=None,**kw):
        super().__init__(*args,**kw)
        self.logos=logos or LogosModule(n_actions=self.n_act)

    def observe(self,o,a,r,no,d):
        super().observe(o,a,r,no,d); self.logos.add_experience(o,a,r)

    def _td_target(self,r_t,no_t,d_t,no_np=None,a_np=None):
        with torch.no_grad(): qn=self.qt(no_t).max(1)[0]
        phi=torch.zeros_like(r_t)
        if no_np is not None:
            for i,nobs in enumerate(no_np):
                ba=int(qn[i].item())
                phi[i]=self.logos.phi(nobs,ba)
        return r_t+self.gamma*(qn+phi)*(1-d_t)

    def state_dict_export(self):
        sd=super().state_dict_export()
        with self.logos._lock: sd['logos_rules']=list(self.logos._rules); sd['logos_UL']=self.logos._UL
        return sd


class MLPFullAgent(MLPBellmanAgent):
    """MLP-Full: Logos in TD target AND action selection."""
    def select_action(self,obs,exploit=False):
        if not exploit and random.random()<self.epsilon: return random.randrange(self.n_act)
        t=torch.tensor(obs,dtype=torch.float32,device=self.dev).unsqueeze(0)
        with torch.no_grad(): q=self.q(t).squeeze(0).cpu().numpy()
        q=self.logos.deduction_value(obs,q)
        return int((q+self.logos.phi_all(obs)).argmax())

    def select_action_explain(self,obs,exploit=False):
        """Returns (action, q_vals, phi_arr, matched_rules) for video overlay."""
        if not exploit and random.random()<self.epsilon:
            return random.randrange(self.n_act),None,None,[]
        t=torch.tensor(obs,dtype=torch.float32,device=self.dev).unsqueeze(0)
        with torch.no_grad(): q=self.q(t).squeeze(0).cpu().numpy()
        q=self.logos.deduction_value(obs,q)
        phi_arr=self.logos.phi_all(obs)
        action=int((q+phi_arr).argmax())
        _,rules=self.logos.explain(obs,action)
        return action,q,phi_arr,rules

print("All agent classes defined ✅")
AGENT_CLASSES = {'MDP':DQNAgent,'MLP-Bellman':MLPBellmanAgent,'MLP-Full':MLPFullAgent}

## 6 · Training Loop

In [ ]:
def make_env(env_cfg):
    """Create LunarLander-v3 with optional wind settings."""
    return gym.make('LunarLander-v3', **env_cfg.get('kwargs',{}))

def evaluate_agent(agent, env_cfg, n_ep=50, seed_offset=9999):
    """Run n_ep greedy episodes. Returns list of episode rewards."""
    rewards=[]
    env=make_env(env_cfg)
    for ep in range(n_ep):
        obs,_=env.reset(seed=seed_offset+ep)
        total=0.0; done=False
        while not done:
            action=agent.select_action(obs,exploit=True)
            obs,r,term,trunc,_=env.step(action)
            total+=r; done=term or trunc
        rewards.append(total)
    env.close()
    return rewards

def train_one_seed(agent_type, exp_key, env_cfg, seed):
    """Train a single (agent_type, env, seed) combination. Returns result dict."""
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    env=make_env(env_cfg)

    agent=AGENT_CLASSES[agent_type](obs_dim=8,n_actions=4)
    agent._ep=0

    ep_rewards=[]; t0=time.time()

    for ep in range(N_EP):
        if over_budget():
            print(f"  ⏰ Budget exceeded at episode {ep}, stopping early")
            break
        obs,_=env.reset(seed=seed*1000+ep)
        total=0.0; done=False
        while not done:
            action=agent.select_action(obs)
            next_obs,reward,term,trunc,_=env.step(action)
            done=term or trunc
            agent.observe(obs,action,reward,next_obs,done)
            agent.update()
            obs=next_obs; total+=reward
        ep_rewards.append(total)
        agent._ep+=1

    env.close()

    # Greedy evaluation
    eval_rewards=evaluate_agent(agent,env_cfg,n_ep=EVAL_EP,seed_offset=seed*10000)

    # Compute metrics
    ma=compute_ma(ep_rewards)
    final_ma100=float(ma[-1]) if len(ma)>0 else float('nan')
    ep_solve=episodes_to_solve(ep_rewards)
    eval_mean=float(np.mean(eval_rewards))
    eval_std =float(np.std(eval_rewards))
    train_min=(time.time()-t0)/60

    result={
        'ep_rewards':  ep_rewards,
        'eval_rewards':eval_rewards,
        'final_ma100': final_ma100,
        'eval_mean':   eval_mean,
        'eval_std':    eval_std,
        'ep_to_solve': ep_solve,
        'train_time_min': train_min,
        'agent_type':  agent_type,
        'exp_key':     exp_key,
        'seed':        seed,
        'n_rules_final': getattr(getattr(agent,'logos',None),'n_rules',0),
        'UL_final':      float(getattr(getattr(agent,'logos',None),'UL',0.0)),
    }

    # Save weights
    sd=agent.state_dict_export()
    save_model(PATHS, sd, agent_type, exp_key, seed)

    return result, agent

print("Training functions defined ✅")

## 7 · Run All Experiments

In [ ]:
# ── Main training loop (auto-resumes from checkpoints) ───────────────────────
ALL_RESULTS = load_all_seed_results(PATHS)
print("Starting / resuming training...")
print_progress(ALL_RESULTS, list(EXPERIMENT_ENVS), AGENT_TYPES, SEEDS)
print()

best_agents = {}   # (agent_type, exp_key) → agent (for best-model export)

for exp_key, env_cfg in EXPERIMENT_ENVS.items():
    for agent_type in AGENT_TYPES:
        best_eval = -float('inf')
        best_sd   = None

        for seed in SEEDS:
            if over_budget():
                print("⏰ Wall-clock budget exceeded — save and resume next session.")
                break

            if is_done(PATHS, exp_key, agent_type, seed):
                res = ALL_RESULTS[exp_key][agent_type][seed]
                print(f"  ↩  {exp_key} / {agent_type} / seed {seed} — cached "
                      f"eval_mean={res['eval_mean']:.1f}")
                if res['eval_mean'] > best_eval:
                    best_eval = res['eval_mean']
                continue

            print(f"  ▶  {exp_key} / {agent_type} / seed {seed} ...", end=' ', flush=True)
            t0 = time.time()
            result, agent = train_one_seed(agent_type, exp_key, env_cfg, seed)
            elapsed = time.time()-t0

            save_seed_result(PATHS, exp_key, agent_type, seed, result)
            ALL_RESULTS.setdefault(exp_key,{}).setdefault(agent_type,{})[seed] = result
            print(f"eval={result['eval_mean']:.1f}±{result['eval_std']:.1f}  "
                  f"ma100={result['final_ma100']:.1f}  "
                  f"ep_solve={result['ep_to_solve']:.0f}  "
                  f"t={elapsed/60:.1f}min")

            if result['eval_mean'] > best_eval:
                best_eval = result['eval_mean']
                best_sd   = agent.state_dict_export()

        if best_sd is not None:
            save_best_model(PATHS, best_sd, agent_type, exp_key)
            print(f"  💾 Best model saved: {agent_type} / {exp_key}  eval={best_eval:.1f}")

print()
print("═══ Final Progress ═══════════════════════════════")
print_progress(ALL_RESULTS, list(EXPERIMENT_ENVS), AGENT_TYPES, SEEDS)

## 8 · Quick Summary Table

In [ ]:
# Print a quick results table (full stats in 02_eval.ipynb)
print(f"{'Env':<10} {'Agent':<15} {'Eval Mean':>10} {'Eval Std':>9} {'MA-100':>8} {'Ep/Solve':>9}")
print('─'*65)
for ek in EXPERIMENT_ENVS:
    for at in AGENT_TYPES:
        seeds_res = ALL_RESULTS.get(ek,{}).get(at,{})
        if not seeds_res: continue
        evals  = [v['eval_mean']   for v in seeds_res.values()]
        ma100s = [v['final_ma100'] for v in seeds_res.values()]
        solves = [v['ep_to_solve'] for v in seeds_res.values() if v['ep_to_solve']<float('inf')]
        print(f"{ek:<10} {at:<15} {np.mean(evals):>10.1f} {np.std(evals):>9.1f} "
              f"{np.mean(ma100s):>8.1f} {np.mean(solves) if solves else float('inf'):>9.1f}")
    print()

print()
print("✅ Training complete. Run 02_eval.ipynb for full statistics and figures.")
print("✅ Run 03_video.ipynb for side-by-side comparison videos.")
print(f"✅ Model weights saved in: {PATHS['models']}")